In [1]:
import asyncio
from datetime import datetime, timedelta, timezone
from decimal import Decimal
import os

from typing_extensions import Literal

from dotenv import load_dotenv

from typed_mexc import MEXC
from typed_mexc.core import ApiError, timestamp_millis as ts
from typed_mexc.futures.http.trade.submit_order import IsolatedMarginOrder
from typed_mexc.futures.streams.market.depth import DepthPushMessage
from typed_mexc.futures.streams.user.my_trades import OrderDealPushMessage
from typed_mexc.spot.http.trade.place_order import (
  LimitOrderRequest,
  MarketOrderByQuantityRequest,
)
from typed_mexc.spot.streams.core.proto import (
  PrivateDealsV3Api,
  PublicLimitDepthsV3Api,
)

from tribulnation.sdk.market import (
  Book,
  Collateral,
  PerpCollateral,
  FundingPayment,
  FundingRate,
  NextFunding,
  Order,
  OrderResponse,
  OrderState,
  PerpPosition,
  Position,
  Rules,
  Trade,
)

load_dotenv()

client = await MEXC.new(
  api_key=os.environ['MEXC_API_KEY'],
  api_secret=os.environ['MEXC_API_SECRET'],
).__aenter__()

MARKETS = {
  'spot': ['BTCUSDT', 'ETHUSDT', 'SOLUSDT'],
  'perp': ['BTC_USDT', 'ETH_USDT', 'SOL_USDT'],
}

## `Market` (spot)

In [2]:
async def depth(symbol: str, *, levels: int | None = None) -> Book:
  raw = await client.spot.http.market.depth(symbol=symbol, limit=levels)
  return Book(
    bids=[Book.Entry(Decimal(p), Decimal(q)) for p, q in raw['bids']],
    asks=[Book.Entry(Decimal(p), Decimal(q)) for p, q in raw['asks']],
  )


{symbol: await depth(symbol, levels=5) for symbol in MARKETS['spot']}

{'BTCUSDT': Book(bids=[Book.Entry(price=Decimal('79630.52'), qty=Decimal('0.00696363')), Book.Entry(price=Decimal('79630.51'), qty=Decimal('0.00132927')), Book.Entry(price=Decimal('79630.32'), qty=Decimal('0.00011200')), Book.Entry(price=Decimal('79630.31'), qty=Decimal('0.04922400')), Book.Entry(price=Decimal('79629.94'), qty=Decimal('0.04885272'))], asks=[Book.Entry(price=Decimal('79630.53'), qty=Decimal('1.16997400')), Book.Entry(price=Decimal('79630.54'), qty=Decimal('0.00126005')), Book.Entry(price=Decimal('79630.55'), qty=Decimal('0.00131499')), Book.Entry(price=Decimal('79630.97'), qty=Decimal('0.00127751')), Book.Entry(price=Decimal('79631.53'), qty=Decimal('0.00122885'))]),
 'ETHUSDT': Book(bids=[Book.Entry(price=Decimal('2478.92'), qty=Decimal('8.79957')), Book.Entry(price=Decimal('2478.91'), qty=Decimal('0.03918')), Book.Entry(price=Decimal('2478.90'), qty=Decimal('0.03713')), Book.Entry(price=Decimal('2478.89'), qty=Decimal('0.04002')), Book.Entry(price=Decimal('2478.86'), 

In [3]:
def depth_stream(symbol: str, *, levels: Literal[5, 10, 20] = 5):
  def to_book(msg: PublicLimitDepthsV3Api) -> Book:
    return Book(
      bids=[Book.Entry(Decimal(e.price), Decimal(e.quantity)) for e in msg.bids],
      asks=[Book.Entry(Decimal(e.price), Decimal(e.quantity)) for e in msg.asks],
    )

  return client.spot.streams.market.depth(symbol, levels).map(to_book)


books: list[Book] = []
async with depth_stream('BTCUSDT') as stream:
  async for book in stream:
    books.append(book)
    if len(books) >= 3:
      break
books

[Book(bids=[Book.Entry(price=Decimal('79630.52'), qty=Decimal('0.00696363')), Book.Entry(price=Decimal('79630.51'), qty=Decimal('0.00132927')), Book.Entry(price=Decimal('79630.21'), qty=Decimal('0.04885272')), Book.Entry(price=Decimal('79629.93'), qty=Decimal('7.02293800')), Book.Entry(price=Decimal('79629.92'), qty=Decimal('0.00126100'))], asks=[Book.Entry(price=Decimal('79630.53'), qty=Decimal('1.16997400')), Book.Entry(price=Decimal('79630.54'), qty=Decimal('0.00126005')), Book.Entry(price=Decimal('79630.55'), qty=Decimal('0.00131499')), Book.Entry(price=Decimal('79630.97'), qty=Decimal('0.00127751')), Book.Entry(price=Decimal('79631.53'), qty=Decimal('0.00122885'))]),
 Book(bids=[Book.Entry(price=Decimal('79630.52'), qty=Decimal('0.00696363')), Book.Entry(price=Decimal('79630.51'), qty=Decimal('0.00132927')), Book.Entry(price=Decimal('79630.21'), qty=Decimal('0.04885272')), Book.Entry(price=Decimal('79629.93'), qty=Decimal('7.02293800')), Book.Entry(price=Decimal('79629.92'), qty=D

In [4]:
async def rules(symbol: str) -> Rules:
  info = await client.spot.http.market.exchange_info(symbol=symbol)
  sym = info['symbols'][0]
  fee = await client.spot.http.account.trade_fee(symbol=symbol)
  # MEXC's exchange_info carries no PRICE_FILTER/LOT_SIZE-style filters (its one spot
  # filter is PERCENT_PRICE_BY_SIDE) -- tick/step size are instead derived from the
  # quote/base precision and quantity-precision fields it does report.
  base_size_precision = sym.get('baseSizePrecision')
  return Rules(
    base=sym['baseAsset'],
    quote=sym['quoteAsset'],
    fee_asset=sym['quoteAsset'],
    tick_size=Decimal(1).scaleb(-sym['quoteAssetPrecision']),
    step_size=Decimal(str(base_size_precision))
    if base_size_precision
    else Decimal(1).scaleb(-sym['baseAssetPrecision']),
    maker_fee=Decimal(str(fee['data']['makerCommission'])),
    taker_fee=Decimal(str(fee['data']['takerCommission'])),
    api=sym['isSpotTradingAllowed'],
    details=sym,
  )


{symbol: await rules(symbol) for symbol in MARKETS['spot']}

{'BTCUSDT': Rules(base='BTC', quote='USDT', fee_asset='USDT', tick_size=Decimal('0.01'), step_size=Decimal('0.000001'), fixed_min_qty=None, min_value=None, max_qty=None, fixed_min_price=None, rel_min_price=None, rel_max_price=None, fixed_max_price=None, maker_fee=Decimal('0.0'), taker_fee=Decimal('0.0005'), api=True, details={'symbol': 'BTCUSDT', 'status': '1', 'baseAsset': 'BTC', 'quoteAsset': 'USDT', 'quotePrecision': 2, 'baseSizePrecision': '0.000001', 'makerCommission': Decimal('0'), 'takerCommission': Decimal('0.0005'), 'orderTypes': ['LIMIT', 'MARKET', 'LIMIT_MAKER'], 'filters': [{'filterType': 'PERCENT_PRICE_BY_SIDE', 'askMultiplierDown': '0.005', 'bidMultiplierUp': '0.005'}], 'isSpotTradingAllowed': True, 'isMarginTradingAllowed': False, 'baseAssetPrecision': 8, 'fullName': 'Bitcoin', 'permissions': ['SPOT'], 'quoteAssetPrecision': 2, 'tradeSideType': 1}),
 'ETHUSDT': Rules(base='ETH', quote='USDT', fee_asset='USDT', tick_size=Decimal('0.01'), step_size=Decimal('0.0001'), fixed

`open_orders`/`trades_history` fail live with the same MEXC error -- this API key
doesn't have the scope for these account-sensitive spot endpoints, confirmed by the
response code, not a code issue:

In [5]:
async def open_orders(symbol: str) -> list[OrderState]:
  raw = await client.spot.http.account.open_orders(symbol=symbol)
  out: list[OrderState] = []
  for o in raw:
    orig_qty = Decimal(o['origQty'])
    executed_qty = Decimal(o['executedQty'])
    sign = 1 if o['side'] == 'BUY' else -1
    out.append(
      OrderState(
        id=str(o['orderId']),
        price=Decimal(o['price']),
        qty=sign * orig_qty,
        filled_qty=sign * executed_qty,
        active=o['status'] in ('NEW', 'PARTIALLY_FILLED'),
        details=o,
      )
    )
  return out


try:
  open_orders_result = {symbol: await open_orders(symbol) for symbol in MARKETS['spot']}
except ApiError as e:
  open_orders_result = e
open_orders_result

typed_core.exceptions.BadRequest(400,
                                 {'code': 700007,
                                  'msg': 'No permission to access the endpoint.'})

In [6]:
async def trades_history(symbol: str, start: datetime, end: datetime) -> list[Trade]:
  raw = await client.spot.http.account.trades(
    symbol=symbol, start_time=start, end_time=end
  )
  return [
    Trade(
      id=str(t['id']),
      price=Decimal(t['price']),
      qty=Decimal(t['qty']) if t['isBuyer'] else -Decimal(t['qty']),
      time=t['time'],
      maker=t['isMaker'],
      fee=Trade.Fee(amount=Decimal(t['commission']), asset=t['commissionAsset'])
      if Decimal(t['commission'])
      else None,
      details=t,
    )
    for t in raw
  ]


end = datetime.now(timezone.utc)
start = end - timedelta(hours=24)
try:
  trades_history_result = {
    symbol: await trades_history(symbol, start, end) for symbol in MARKETS['spot']
  }
except ApiError as e:
  trades_history_result = e
trades_history_result

typed_core.exceptions.BadRequest(400,
                                 {'code': 700007,
                                  'msg': 'No permission to access the endpoint.'})

`trades_stream` (the account's own private fills feed) doesn't need the same scope and
works live -- it simply observes no fills in the window. Note MEXC's private-deals push
(`spot@private.deals.v3.api.pb`) carries no symbol field (unlike the order-update push,
which has `market`), so unlike other venues this can't be filtered by symbol at all --
it's account-wide by construction:

In [7]:
def trades_stream():
  def to_trade(d: PrivateDealsV3Api) -> Trade:
    qty = Decimal(d.quantity)
    side = 'BUY' if d.trade_type == 1 else 'SELL'
    return Trade(
      id=d.trade_id,
      price=Decimal(d.price),
      qty=qty if side == 'BUY' else -qty,
      time=ts.parse(d.time),
      maker=d.is_maker,
      fee=Trade.Fee(amount=Decimal(d.fee_amount), asset=d.fee_currency)
      if d.fee_amount
      else None,
      details=d,
    )

  return client.spot.streams.user.trades().map(to_trade)


async with trades_stream() as stream:
  it = aiter(stream)
  try:
    trade = await asyncio.wait_for(anext(it), timeout=5.0)
  except asyncio.TimeoutError:
    trade = 'no new trades observed in 5s (expected -- no live trading on this account)'
trade

'no new trades observed in 5s (expected -- no live trading on this account)'

In [8]:
async def position(symbol: str) -> Position:
  info = await client.spot.http.market.exchange_info(symbol=symbol)
  base = info['symbols'][0]['baseAsset']
  account = await client.spot.http.account.info()
  balance = next((b for b in account['balances'] if b['asset'] == base), None)
  size = (
    (Decimal(balance['free']) + Decimal(balance['locked'])) if balance else Decimal(0)
  )
  return Position(size=size)


{symbol: await position(symbol) for symbol in MARKETS['spot']}

Task was destroyed but it is pending!
task: <Task pending name='Task-60' coro=<Queue.get() done, defined at /home/ubuntu/.local/share/uv/python/cpython-3.12.13-linux-x86_64-gnu/lib/python3.12/asyncio/queues.py:149> wait_for=<Future cancelled>>


{'BTCUSDT': Position(size=Decimal('0.0000012814')),
 'ETHUSDT': Position(size=Decimal('0')),
 'SOLUSDT': Position(size=Decimal('0'))}

In [9]:
async def collateral(symbol: str) -> Collateral:
  info = await client.spot.http.market.exchange_info(symbol=symbol)
  quote = info['symbols'][0]['quoteAsset']
  account = await client.spot.http.account.info()
  balance = next((b for b in account['balances'] if b['asset'] == quote), None)
  free = Decimal(balance['free']) if balance else Decimal(0)
  locked = Decimal(balance['locked']) if balance else Decimal(0)
  return Collateral(equity=free + locked, free_collateral=free)


{symbol: await collateral(symbol) for symbol in MARKETS['spot']}

{'BTCUSDT': Collateral(equity=Decimal('0.0278906756888'), free_collateral=Decimal('0.0278906756888')),
 'ETHUSDT': Collateral(equity=Decimal('0.0278906756888'), free_collateral=Decimal('0.0278906756888')),
 'SOLUSDT': Collateral(equity=Decimal('0.0278906756888'), free_collateral=Decimal('0.0278906756888'))}

In [10]:
async def available_notional(symbol: str) -> Decimal:
  c = await collateral(symbol)
  return c.free_collateral


{symbol: await available_notional(symbol) for symbol in MARKETS['spot']}

{'BTCUSDT': Decimal('0.0278906756888'),
 'ETHUSDT': Decimal('0.0278906756888'),
 'SOLUSDT': Decimal('0.0278906756888')}

### Mutating (written, never executed)

In [ ]:
async def place_order(symbol: str, order: Order) -> OrderResponse:
  qty = Decimal(order['qty'])
  side: Literal['BUY', 'SELL'] = 'BUY' if qty > 0 else 'SELL'
  quantity = abs(qty)
  price = Decimal(order['price'])
  body: LimitOrderRequest | MarketOrderByQuantityRequest
  if order['type'] == 'MARKET':
    body = {'symbol': symbol, 'side': side, 'type': 'MARKET', 'quantity': quantity}
  elif order['type'] == 'POST_ONLY':
    body = {
      'symbol': symbol,
      'side': side,
      'type': 'LIMIT_MAKER',
      'quantity': quantity,
      'price': price,
    }
  else:
    body = {
      'symbol': symbol,
      'side': side,
      'type': 'LIMIT',
      'quantity': quantity,
      'price': price,
    }
  raw = await client.spot.http.trade.place_order(body)
  return OrderResponse(id=str(raw['orderId']), details=raw)


# Not executed here -- would place a real order on the account.
await place_order(
  'BTCUSDT', {'qty': Decimal('0.0001'), 'price': Decimal('20000'), 'type': 'LIMIT'}
)

In [ ]:
async def cancel_order(symbol: str, id: str):
  return await client.spot.http.trade.cancel_order(symbol=symbol, order_id=id)


# Not executed here -- would cancel a real order on the account.
await cancel_order('BTCUSDT', '123456')

### Coverage assessment: `Market` (spot)

**Fully supported for the abstract interface's read side**, live-tested end-to-end
above: `depth`/`depth_stream`, `rules`, `position`/`collateral`/`available_notional`.
`open_orders`/`trades_history` both fail live with the same MEXC error (`700007 No
permission to access the endpoint`) -- a scope this API key doesn't have, not a code
issue; `trades_stream` (the account's own private-fills feed) works and simply observes
no fills in the 5-second window, though see the note above the cell: MEXC's private
deals push has no symbol field at all, so `trades_stream` can't be scoped to one market
the way `depth_stream`/`open_orders`/`trades_history` are -- it's inherently
account-wide, unlike the sibling venues explored in `../../kucoin/poc/market.ipynb` and
`../../binance/poc/market.ipynb`, both of which filter their user-trade push by symbol.
`place_order`/`cancel_order` are written but never executed, since they would mutate the
real account.

`rules()` also differs structurally from Binance/KuCoin: MEXC's `exchange_info` carries
no `PRICE_FILTER`/`LOT_SIZE`-style filter list for spot symbols (its one filter here is
`PERCENT_PRICE_BY_SIDE`), so `tick_size`/`step_size` are derived from the flat
`quoteAssetPrecision`/`baseSizePrecision` fields instead -- `fixed_min_qty`/`min_value`/
`max_qty` are left `None` since MEXC's spot market data reports none of them.

## `PerpMarket` (USDT-M futures)

In [11]:
async def perp_depth(symbol: str, *, levels: int | None = None) -> Book:
  raw = await client.futures.http.market.depth(symbol=symbol, limit=levels)
  data = raw.get('data')
  assert data is not None, f'no depth data for {symbol}: {raw}'
  return Book(
    bids=[Book.Entry(Decimal(str(p)), Decimal(str(q))) for p, q, _ in data['bids']],
    asks=[Book.Entry(Decimal(str(p)), Decimal(str(q))) for p, q, _ in data['asks']],
  )


{symbol: await perp_depth(symbol, levels=5) for symbol in MARKETS['perp']}

{'BTC_USDT': Book(bids=[Book.Entry(price=Decimal('79553.1'), qty=Decimal('5117.0')), Book.Entry(price=Decimal('79553.0'), qty=Decimal('5035.0')), Book.Entry(price=Decimal('79552.9'), qty=Decimal('5050.0')), Book.Entry(price=Decimal('79552.8'), qty=Decimal('7536.0')), Book.Entry(price=Decimal('79552.7'), qty=Decimal('5056.0'))], asks=[Book.Entry(price=Decimal('79553.2'), qty=Decimal('17200.0')), Book.Entry(price=Decimal('79556.9'), qty=Decimal('1892.0')), Book.Entry(price=Decimal('79557.6'), qty=Decimal('1850.0')), Book.Entry(price=Decimal('79557.8'), qty=Decimal('1850.0')), Book.Entry(price=Decimal('79560.1'), qty=Decimal('1892.0'))]),
 'ETH_USDT': Book(bids=[Book.Entry(price=Decimal('2476.88'), qty=Decimal('28176.0')), Book.Entry(price=Decimal('2476.87'), qty=Decimal('49.0')), Book.Entry(price=Decimal('2476.86'), qty=Decimal('41.0')), Book.Entry(price=Decimal('2476.85'), qty=Decimal('43.0')), Book.Entry(price=Decimal('2476.84'), qty=Decimal('44.0'))], asks=[Book.Entry(price=Decimal('2

In [12]:
def perp_depth_stream(symbol: str):
  def to_book(msg: DepthPushMessage) -> Book:
    data = msg['data']
    return Book(
      bids=[Book.Entry(Decimal(str(p)), Decimal(str(q))) for p, q, _ in data['bids']],
      asks=[Book.Entry(Decimal(str(p)), Decimal(str(q))) for p, q, _ in data['asks']],
    )

  return client.futures.streams.market.depth(symbol).map(to_book)


books: list[Book] = []
async with perp_depth_stream('BTC_USDT') as stream:
  async for book in stream:
    books.append(book)
    if len(books) >= 3:
      break
books

[Book(bids=[Book.Entry(price=Decimal('79553.1'), qty=Decimal('36941.0')), Book.Entry(price=Decimal('77166.6'), qty=Decimal('0.0'))], asks=[Book.Entry(price=Decimal('79553.2'), qty=Decimal('2135.0')), Book.Entry(price=Decimal('79555.0'), qty=Decimal('0.0')), Book.Entry(price=Decimal('79588.2'), qty=Decimal('0.0'))]),
 Book(bids=[Book.Entry(price=Decimal('79553.1'), qty=Decimal('13379.0')), Book.Entry(price=Decimal('79552.3'), qty=Decimal('5035.0')), Book.Entry(price=Decimal('79552.2'), qty=Decimal('5090.0'))], asks=[Book.Entry(price=Decimal('79553.2'), qty=Decimal('34162.0')), Book.Entry(price=Decimal('79554.8'), qty=Decimal('0.0')), Book.Entry(price=Decimal('79554.9'), qty=Decimal('0.0')), Book.Entry(price=Decimal('79555.8'), qty=Decimal('1850.0')), Book.Entry(price=Decimal('79555.9'), qty=Decimal('1850.0')), Book.Entry(price=Decimal('79556.4'), qty=Decimal('1850.0')), Book.Entry(price=Decimal('79556.5'), qty=Decimal('2590.0')), Book.Entry(price=Decimal('79556.6'), qty=Decimal('0.0')),

In [13]:
async def perp_rules(symbol: str) -> Rules:
  raw = await client.futures.http.market.contract_info(symbol=symbol)
  spec = raw.get('data')
  assert spec is not None and not isinstance(spec, list), (
    f'no contract spec for {symbol}: {raw}'
  )
  return Rules(
    base=spec['baseCoin'],
    quote=spec['quoteCoin'],
    fee_asset=spec['settleCoin'],
    tick_size=Decimal(str(spec['priceUnit'])),
    step_size=Decimal(str(spec['volUnit'])),
    fixed_min_qty=Decimal(str(spec['minVol'])),
    max_qty=Decimal(str(spec['maxVol'])),
    maker_fee=Decimal(str(spec['makerFeeRate'])),
    taker_fee=Decimal(str(spec['takerFeeRate'])),
    api=spec['apiAllowed'],
    details=spec,
  )


{symbol: await perp_rules(symbol) for symbol in MARKETS['perp']}

{'BTC_USDT': Rules(base='BTC', quote='USDT', fee_asset='USDT', tick_size=Decimal('0.1'), step_size=Decimal('1.0'), fixed_min_qty=Decimal('1.0'), min_value=None, max_qty=Decimal('400000.0'), fixed_min_price=None, rel_min_price=None, rel_max_price=None, fixed_max_price=None, maker_fee=Decimal('0.0'), taker_fee=Decimal('0.0002'), api=True, details={'symbol': 'BTC_USDT', 'displayName': 'BTC_USDT永续', 'displayNameEn': 'BTC_USDT PERPETUAL', 'positionOpenType': 3, 'baseCoin': 'BTC', 'quoteCoin': 'USDT', 'settleCoin': 'USDT', 'contractSize': 0.0001, 'minLeverage': 1, 'maxLeverage': 500, 'priceScale': 1, 'volScale': 0, 'amountScale': 4, 'priceUnit': 0.1, 'volUnit': 1.0, 'minVol': 1.0, 'maxVol': 400000.0, 'state': 0, 'apiAllowed': True, 'appraisal': 0, 'askLimitPriceRate': 0.1, 'automaticDelivery': 0, 'baseCoinIconUrl': 'https://public.mocortech.com/coin/F20250612182226438Ba037qttKoGcrm.png', 'baseCoinId': 'febc9973be4d4d53bb374476239eb219', 'baseCoinName': 'BTC', 'bidLimitPriceRate': 0.1, 'conce

`open_orders`/`trades_history`/`position`/`collateral`/`available_notional` all fail
live with the same MEXC futures error -- this API key isn't enabled for futures account
read access, confirmed by the response code, not a code issue:

In [14]:
async def perp_open_orders(symbol: str) -> list[OrderState]:
  raw = (
    await client.futures.http.trade.open_orders(symbol, page_num=1, page_size=100)
  ).get('data') or []
  out: list[OrderState] = []
  for o in raw:
    vol = Decimal(str(o['vol']))
    sign = (
      1 if o['side'] in (1, 2) else -1
    )  # 1 open long, 2 close short -> net buy direction
    out.append(
      OrderState(
        id=str(o['orderId']),
        price=Decimal(str(o['price'])),
        qty=sign * vol,
        filled_qty=sign * Decimal(str(o['dealVol'])),
        active=o['state'] in (1, 2),
        details=o,
      )
    )
  return out


try:
  perp_open_orders_result = {
    symbol: await perp_open_orders(symbol) for symbol in MARKETS['perp']
  }
except ApiError as e:
  perp_open_orders_result = e
perp_open_orders_result

typed_core.exceptions.AuthError(703,
                                'Trading information read access is required',
                                {'success': False,
                                 'code': 703,
                                 'message': 'Trading information read access is required'})

In [ ]:
async def perp_trades_history(
  symbol: str, start: datetime, end: datetime
) -> list[Trade]:
  raise NotImplementedError(
    'blocked: typed_mexc declares OrderDeal with both isTaker and taker (requiredness '
    'inverted between order_deals and deal_details) and timestamp as int | str, so '
    'neither the maker flag nor the fill time can be read without guessing the wire '
    'shape'
  )
  raw = (
    await client.futures.http.trade.order_deals(
      symbol=symbol,
      start_time=start,
      end_time=end,
      page_num=1,
      page_size=100,
    )
  ).get('data') or []
  out: list[Trade] = []
  for d in raw:
    vol = Decimal(str(d['vol']))
    sign = 1 if d['side'] in (1, 2) else -1
    out.append(
      Trade(
        id=str(d['id']),
        price=Decimal(str(d['price'])),
        qty=sign * vol,
        time=d['timestamp'],
        maker=not d['isTaker'],
        fee=Trade.Fee(amount=Decimal(str(d['fee'])), asset=d['feeCurrency'])
        if d['fee']
        else None,
        details=d,
      )
    )
  return out


# not executed: blocked by typed-mexc "OrderDeal declares two spellings of the taker flag" and
# "Futures timestamps are declared int | str"; the API key also lacks futures read scope (703)
end = datetime.now(timezone.utc)
start = end - timedelta(hours=24)
{symbol: await perp_trades_history(symbol, start, end) for symbol in MARKETS['perp']}

`perp_trades_stream` (the account's own fill feed, `personal.order.deal`) authenticates
at the WebSocket layer independently of the REST futures-read scope above, so unlike the
REST calls it connects live and simply observes no fills in the window:

In [15]:
def perp_trades_stream(symbol: str):
  def to_trade(msg: OrderDealPushMessage) -> Trade:
    d = msg['data']
    vol = Decimal(str(d['vol']))
    sign = 1 if d['side'] in (1, 2) else -1
    return Trade(
      id=str(d['id']),
      price=Decimal(str(d['price'])),
      qty=sign * vol,
      time=d['timestamp'],
      maker=not d['taker'],
      fee=Trade.Fee(amount=Decimal(str(d['fee'])), asset=d['feeCurrency'])
      if d['fee']
      else None,
      details=d,
    )

  return (
    client.futures.streams.user.my_trades()
    .filter(lambda msg: msg['data']['symbol'] == symbol)
    .map(to_trade)
  )


async with perp_trades_stream('BTC_USDT') as stream:
  it = aiter(stream)
  try:
    trade = await asyncio.wait_for(anext(it), timeout=5.0)
  except asyncio.TimeoutError:
    trade = 'no new trades observed in 5s (expected -- no live trading on this account)'
trade

'no new trades observed in 5s (expected -- no live trading on this account)'

In [16]:
async def perp_position(symbol: str) -> PerpPosition:
  raw = (await client.futures.http.position.open(symbol=symbol)).get('data') or []
  if not raw:
    return PerpPosition()
  p = raw[0]
  sign = 1 if p['positionType'] == 1 else -1
  return PerpPosition(
    size=sign * Decimal(str(p['holdVol'])),
    entry_price=Decimal(str(p['holdAvgPrice'])),
  )


try:
  perp_position_result = {
    symbol: await perp_position(symbol) for symbol in MARKETS['perp']
  }
except ApiError as e:
  perp_position_result = e
perp_position_result

typed_core.exceptions.AuthError(703,
                                'Trading information read access is required',
                                {'success': False,
                                 'code': 703,
                                 'message': 'Trading information read access is required'})

In [ ]:
async def perp_collateral(symbol: str) -> PerpCollateral:
  raise NotImplementedError(
    'not supported: MEXC reports no maintenance-margin figure -- neither '
    'futures.http.account.assets nor futures.http.position.open carries one, and '
    'PerpCollateral.maintenance_margin is required'
  )
  contract = await client.futures.http.market.contract_info(symbol=symbol)
  spec = contract.get('data')
  assert spec is not None and not isinstance(spec, list), (
    f'no contract spec for {symbol}: {contract}'
  )
  assets = (await client.futures.http.account.assets()).get('data') or []
  positions = (await client.futures.http.position.open(symbol=symbol)).get('data') or []
  row = next((a for a in assets if a['currency'] == spec['settleCoin']), None)
  equity = Decimal(str(row['equity'])) if row else Decimal(0)
  free_collateral = Decimal(str(row['availableBalance'])) if row else Decimal(0)
  position = positions[0] if positions else None
  if position is not None:
    notional = (
      Decimal(str(position['holdVol']))
      * Decimal(str(position['holdAvgPrice']))
      * Decimal(str(spec['contractSize']))
    )
    initial_margin = Decimal(str(position['im']))
    maintenance_margin = Decimal(0)
    leverage = notional / equity if equity > 0 else Decimal(0)
    margin_mode = 'isolated' if position['openType'] == 1 else 'cross'
  else:
    initial_margin = Decimal(0)
    maintenance_margin = Decimal(0)
    leverage = Decimal(0)
    margin_mode = 'cross'
  return PerpCollateral(
    equity=equity,
    free_collateral=free_collateral,
    initial_margin=initial_margin,
    maintenance_margin=maintenance_margin,
    leverage=leverage,
    margin_mode=margin_mode,
  )


# not executed: not supported -- no venue-reported maintenance margin (see the coverage note)
{symbol: await perp_collateral(symbol) for symbol in MARKETS['perp']}

In [17]:
async def perp_available_notional(symbol: str) -> Decimal:
  contract = await client.futures.http.market.contract_info(symbol=symbol)
  spec = contract.get('data')
  assert spec is not None and not isinstance(spec, list), (
    f'no contract spec for {symbol}: {contract}'
  )
  assets = (await client.futures.http.account.assets()).get('data') or []
  row = next((a for a in assets if a['currency'] == spec['settleCoin']), None)
  free_collateral = Decimal(str(row['availableBalance'])) if row else Decimal(0)
  return free_collateral * spec['maxLeverage']


try:
  perp_available_notional_result = {
    symbol: await perp_available_notional(symbol) for symbol in MARKETS['perp']
  }
except ApiError as e:
  perp_available_notional_result = e
perp_available_notional_result

typed_core.exceptions.AuthError(701,
                                'Please enable API Key read access',
                                {'success': False,
                                 'code': 701,
                                 'message': 'Please enable API Key read access'})

In [18]:
async def index(symbol: str) -> Decimal:
  raw = await client.futures.http.market.index_price(symbol)
  data = raw.get('data')
  assert data is not None, f'no index price for {symbol}: {raw}'
  return Decimal(str(data['indexPrice']))


{symbol: await index(symbol) for symbol in MARKETS['perp']}

{'BTC_USDT': Decimal('79582.2'),
 'ETH_USDT': Decimal('2477.58'),
 'SOL_USDT': Decimal('105.57')}

In [19]:
async def next_funding(symbol: str) -> NextFunding:
  raw = await client.futures.http.market.funding_rate(symbol)
  data = raw.get('data')
  assert data is not None, f'no funding rate for {symbol}: {raw}'
  return NextFunding(
    rate=Decimal(str(data['fundingRate'])),
    time=data['nextSettleTime'],
    interval=timedelta(hours=data['collectCycle']),
  )


{symbol: await next_funding(symbol) for symbol in MARKETS['perp']}

{'BTC_USDT': NextFunding(rate=Decimal('0.000025'), time=datetime.datetime(2026, 9, 6, 16, 0, tzinfo=datetime.timezone.utc), premium=None, interval=datetime.timedelta(seconds=28800)),
 'ETH_USDT': NextFunding(rate=Decimal('0.000024'), time=datetime.datetime(2026, 9, 6, 16, 0, tzinfo=datetime.timezone.utc), premium=None, interval=datetime.timedelta(seconds=28800)),
 'SOL_USDT': NextFunding(rate=Decimal('0.0001'), time=datetime.datetime(2026, 9, 6, 16, 0, tzinfo=datetime.timezone.utc), premium=None, interval=datetime.timedelta(seconds=28800))}

In [20]:
async def funding_rates(
  symbol: str,
  start: datetime | None = None,
  end: datetime | None = None,
) -> list[FundingRate]:
  # funding_rate_history has no start/end params -- only page_num/page_size, newest
  # settlement first -- so the window is applied client-side while paging back in time.
  out: list[FundingRate] = []
  page_num = 1
  while True:
    raw = await client.futures.http.market.funding_rate_history(
      symbol=symbol,
      page_num=page_num,
      page_size=100,
    )
    page = raw.get('data')
    assert page is not None, f'no funding rate history for {symbol}: {raw}'
    for r in page['resultList']:
      time = r['settleTime']
      if start is not None and time < start:
        return out
      if end is None or time <= end:
        out.append(FundingRate(rate=Decimal(str(r['fundingRate'])), time=time))
    if page_num >= page['totalPage']:
      return out
    page_num += 1


end = datetime.now(timezone.utc)
start = end - timedelta(days=7)
{symbol: await funding_rates(symbol, start, end) for symbol in MARKETS['perp']}

{'BTC_USDT': [FundingRate(rate=Decimal('0.000023'), time=datetime.datetime(2026, 9, 6, 8, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.000036'), time=datetime.datetime(2026, 9, 6, 0, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.000019'), time=datetime.datetime(2026, 9, 5, 16, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('-0.000001'), time=datetime.datetime(2026, 9, 5, 8, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.00001'), time=datetime.datetime(2026, 9, 5, 0, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.000045'), time=datetime.datetime(2026, 9, 4, 16, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.000064'), time=datetime.datetime(2026, 9, 4, 8, 0, tzinfo=datetime.timezone.utc), premium=None),
  FundingRate(rate=Decimal('0.000085'), time=datetime.datetime(2026, 9, 4, 0, 0, tzinfo=datetime.

`funding_payments` fails live with the same futures-read-access error as the other
account-scoped futures endpoints above:

In [21]:
async def funding_payments(
  symbol: str, start: datetime, end: datetime
) -> list[FundingPayment]:
  # MEXC's `funding` field convention (positive = received vs. paid) isn't verifiable
  # live here (blocked by the same futures-read-access scope) -- assumed by symmetry
  # with typed_binance's `income`, i.e. negated to match FundingPayment's
  # "positive = paid" convention.
  # funding_records has no start/end params either -- only page_num/page_size, newest
  # settlement first, same client-side windowing as funding_rates above.
  out: list[FundingPayment] = []
  page_num = 1
  while True:
    raw = await client.futures.http.account.funding_records(
      symbol=symbol,
      page_num=page_num,
      page_size=100,
    )
    page = raw.get('data')
    assert page is not None, f'no funding records for {symbol}: {raw}'
    for r in page['resultList']:
      time = r['settleTime']
      if time < start:
        return out
      if time <= end:
        out.append(FundingPayment(amount=-Decimal(str(r['funding'])), time=time))
    if page_num >= page['totalPage']:
      return out
    page_num += 1


end = datetime.now(timezone.utc)
start = end - timedelta(days=7)
try:
  funding_payments_result = {
    symbol: await funding_payments(symbol, start, end) for symbol in MARKETS['perp']
  }
except ApiError as e:
  funding_payments_result = e
funding_payments_result

typed_core.exceptions.AuthError(703,
                                'Trading information read access is required',
                                {'success': False,
                                 'code': 703,
                                 'message': 'Trading information read access is required'})

### Mutating (written, never executed)

MEXC futures orders need a margin mode (`openType`) and, for isolated margin, a
`leverage` -- neither is part of the abstract `Order`/`Settings` shape yet, so this
demonstration hardcodes 1x isolated margin as an illustrative simplification.

In [ ]:
async def perp_place_order(symbol: str, order: Order) -> OrderResponse:
  contract = await client.futures.http.market.contract_info(symbol=symbol)
  spec = contract.get('data')
  assert spec is not None and not isinstance(spec, list), (
    f'no contract spec for {symbol}: {contract}'
  )
  qty = Decimal(order['qty'])
  side: Literal[1, 3] = 1 if qty > 0 else 3  # 1 open long, 3 open short
  vol = float(abs(qty) / Decimal(str(spec['contractSize'])))
  price = float(Decimal(order['price']))
  order_type: Literal[1, 2, 5]  # 1 limit, 2 post-only, 5 market
  if order['type'] == 'MARKET':
    order_type = 5
  elif order['type'] == 'POST_ONLY':
    order_type = 2
  else:
    order_type = 1
  body: IsolatedMarginOrder = {
    'symbol': symbol,
    'price': price,
    'vol': vol,
    'leverage': 1,
    'side': side,
    'type': order_type,
    'openType': 1,
  }
  raw = await client.futures.http.trade.submit_order(body)
  order_id = raw.get('data')
  assert order_id is not None, f'no order id in response: {raw}'
  return OrderResponse(id=str(order_id), details=raw)


# Not executed here -- would place a real order on the account.
await perp_place_order(
  'BTC_USDT', {'qty': Decimal('0.001'), 'price': Decimal('20000'), 'type': 'LIMIT'}
)

In [ ]:
async def perp_cancel_order(order_id: str):
  return await client.futures.http.trade.cancel_order([order_id])


# Not executed here -- would cancel a real order on the account.
await perp_cancel_order('123456')

### Coverage assessment: `PerpMarket` (USDT-M futures)

**Present and broader than production.** Unlike `tribulnation.mexc.market`, which has no
`PerpExchange`/`PerpMarket` implementation at all (the equivalent legacy PoC notebook's
coverage note flags this as an absent pillar), `typed_mexc` has a complete, independently
transported `futures` surface (`client.futures.http.{market,account,position,trade}`
plus `client.futures.streams`)
covering every method the abstract `PerpMarket` interface needs: `depth`/`depth_stream`,
`rules`, `open_orders`/`trades_history`/`trades_stream`, `position`/`collateral`/
`available_notional`, `index`/`next_funding`/`funding_rates`/`funding_payments`,
`perp_position`/`perp_collateral`, and `place_order`/`cancel_order` (written, never
executed). This is the single biggest surface gap this rewrite found relative to what
the 4 legacy notebooks (importing production `tribulnation.mexc` directly) could cover.

Public futures market data (`depth`/`depth_stream`, `rules` via `contract_info`, `index`,
`next_funding`, `funding_rates`) is **fully live-tested** above. Every account-scoped
REST call (`open_orders`, `trades_history`, `position`/`collateral`/
`available_notional`, `funding_payments`) fails live with `701 Please enable API Key
read access` / `703 Trading information read access is required` -- this API key isn't
enabled for futures account access at all, not a code issue. `perp_trades_stream`
(`personal.order.deal`) is the one account-scoped call that *does* work live despite
that: MEXC's futures user WebSocket authenticates independently of the REST scope, so it
connects and simply observes no fills in the window, mirroring spot's `trades_stream`
above. `place_order`/`cancel_order` are written but never executed.

Two methods are not mapped:
1. `perp_trades_history` is **blocked** on typed-mexc "OrderDeal declares two spellings
   of the taker flag" and "Futures timestamps are declared `int | str`": `order_deals`
   requires `isTaker` and makes `taker` optional while `deal_details` does the opposite,
   and `timestamp` is `int | str`, so the maker flag and fill time cannot be read without
   guessing the wire shape. Its body raises `NotImplementedError` until the client
   settles both; the API key's missing futures read scope (`703`) would block a live
   check anyway.
2. `perp_collateral` is **not supported**: MEXC reports no maintenance-margin figure
   (neither `futures.http.account.assets` nor `futures.http.position.open` carries one),
   and `PerpCollateral.maintenance_margin` is required. `position.leverage` exposes the
   account's current `mmr`, but a figure computed from it would be derived, not read.

One unverified assumption remains: `funding_payments`' sign (`positive = paid`) is
asserted by symmetry with `typed_binance`'s equivalent `income` field, not confirmed
against a live MEXC response, because the account-scoped call fails with `703`.